# CeNN Kernel Variant Laboratory

This notebook asks a narrower question than LM training: **which finite-state attention formulation best preserves frozen SmolLM2 softmax attention?**

Variants compared:
- `current_prf`: existing antithetic positive random features with hard log-feature clipping.
- `stable_prf`: same antithetic random directions, but recurrent running-max normalization and no hard exponent clipping. This isolates numerical normalization.
- `stable_orf`: stable normalization plus block-orthogonal Gaussian random features to reduce Monte-Carlo variance.
- `taylor2`: deterministic second-order kernel, $1+s+s^2/2$, represented exactly by a finite feature map. For head dimension 64 this uses 2145 state features.
- `elu1`: cheap positive ELU+1 linear-attention baseline.

The benchmark keeps the Transformer's Q/K/V projections, RoPE, O projection, RMSNorm and residual path unchanged. It evaluates raw attention fidelity, softmax partition fidelity, attention-distribution KL/JS, top-k overlap, memory break-even, runtime, and optional **Q/K/V gradient fidelity**.


In [ ]:
import pathlib, subprocess, sys

REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/vtavakkoli/TinyCeNN-LM.git', str(REPO_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('Repository ready:', REPO_DIR)


## Experiment configuration
The default run is deliberately small enough for Colab but includes the difficult layers found in the previous preservation benchmark. Gradient checks are more expensive, so they use one feature size and three representative layers.


In [ ]:
from pathlib import Path

CONTEXT_LENGTH = 128
NUM_SEQUENCES = 2
LAYERS = '0,6,14,18,20,23,29'
FEATURE_DIMS = '256,512,1024,2048'
VARIANTS = 'current_prf,stable_prf,stable_orf,taylor2,elu1'
GRADIENT_LAYERS = '0,18,29'
GRADIENT_FEATURE_DIM = 1024
OUTPUT_DIR = Path('/content/cenn-kernel-variants')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Variants:', VARIANTS)
print('Layers:', LAYERS)
print('Random-feature dimensions:', FEATURE_DIMS)


In [ ]:
import subprocess, sys

cmd = [
    sys.executable, str(REPO_DIR / 'scripts' / 'benchmark_cenn_kernel_variants.py'),
    '--base-model', 'HuggingFaceTB/SmolLM2-135M',
    '--context-length', str(CONTEXT_LENGTH),
    '--num-sequences', str(NUM_SEQUENCES),
    '--layers', LAYERS,
    '--variants', VARIANTS,
    '--feature-dims', FEATURE_DIMS,
    '--gradient-check',
    '--gradient-layers', GRADIENT_LAYERS,
    '--gradient-feature-dim', str(GRADIENT_FEATURE_DIM),
    '--output-dir', str(OUTPUT_DIR),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## Main comparison
The heuristic score below is only a sorting aid. A suitable CeNN replacement should improve **all** of these: raw attention cosine/NMSE, partition-function error, attention-distribution KL/top-k overlap, and gradient cosine. Memory and runtime are considered separately.


In [ ]:
import json, pandas as pd
from IPython.display import display

summary = pd.read_csv(OUTPUT_DIR / 'kernel_variant_summary.csv')
report = json.loads((OUTPUT_DIR / 'kernel_variant_report.json').read_text())
cols = [c for c in [
    'variant','requested_feature_dim','effective_feature_dim','output_cosine','output_nmse',
    'partition_log_mae','attention_kl','topk_overlap','grad_mean_cosine','grad_mean_nmse',
    'state_vs_kv_ratio','break_even_tokens','runtime_ms','heuristic_selection_score'
] if c in summary.columns]
display(summary[cols].sort_values('heuristic_selection_score', ascending=False).round(5))
print('Teacher reconstruction sanity:', report['teacher_reconstruction_sanity'])
print('Strict candidates:', report['strict_candidates'])
print('Best by heuristic:', report['best_by_heuristic'])


In [ ]:
import matplotlib.pyplot as plt

plot_df = summary.copy()
plot_df['label'] = plot_df.apply(lambda r: f"{r['variant']}\nF={int(r['effective_feature_dim'])}", axis=1)
plot_df = plot_df.sort_values('output_cosine', ascending=False)
plt.figure(figsize=(12,5))
plt.bar(plot_df['label'], plot_df['output_cosine'])
plt.axhline(0.90, linestyle='--', label='first useful target 0.90')
plt.axhline(0.99, linestyle=':', label='near-equivalence target 0.99')
plt.ylabel('Attention-output cosine')
plt.title('Forward attention preservation by kernel variant')
plt.xticks(rotation=55, ha='right')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11,5))
for variant, part in summary.groupby('variant'):
    part = part.sort_values('effective_feature_dim')
    plt.plot(part['effective_feature_dim'], part['partition_log_mae'], marker='o', label=variant)
plt.xscale('log', base=2)
plt.yscale('log')
plt.xlabel('Effective CeNN feature dimension')
plt.ylabel('|log Z_approx - log Z_softmax|')
plt.title('Does the variant preserve Transformer attention normalization?')
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## Gradient preservation
For the same fixed scalar probe loss, the notebook compares $dL/dQ$, $dL/dK$, and $dL/dV$ from each approximation with exact softmax attention. A high forward cosine with poor gradient cosine is **not** a safe trainable replacement.


In [ ]:
grad_path = OUTPUT_DIR / 'kernel_variant_gradients.csv'
if grad_path.exists():
    grads = pd.read_csv(grad_path)
    display(grads.round(5))
    g = grads.groupby(['variant','effective_feature_dim'], as_index=False)['grad_mean_cosine'].mean().sort_values('grad_mean_cosine', ascending=False)
    plt.figure(figsize=(10,4))
    plt.bar(g.apply(lambda r: f"{r['variant']}\nF={int(r['effective_feature_dim'])}", axis=1), g['grad_mean_cosine'])
    plt.axhline(0.80, linestyle='--', label='useful gradient target')
    plt.ylabel('Mean Q/K/V gradient cosine')
    plt.title('Backpropagation fidelity vs exact softmax attention')
    plt.xticks(rotation=40, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Gradient report not found.')


## Why the stabilization test matters
`current_prf` and `stable_prf` use the same antithetic Gaussian directions. The only intended difference is numerical representation. If `stable_prf` materially improves partition error and gradients, hard exponent clipping / unstable scale was a real part of the problem. If it does not, the dominant problem is estimator variance rather than normalization.

`stable_orf` then asks whether orthogonal directions reduce that variance. `taylor2` is a deterministic control: no Monte-Carlo variance and polynomial gradients, but a larger fixed state. `elu1` shows the quality attainable by a very cheap generic positive feature map that does not attempt to reproduce the softmax kernel exactly.


In [ ]:
score_info = report.get('score_distribution_summary', {})
print('Observed pretrained attention-score distribution:')
for k,v in score_info.items():
    print(f'{k:16s}: {v:.5f}' if isinstance(v, (int,float)) else k, v)

clip_cols = [c for c in ['variant','effective_feature_dim','upper_clip_fraction','lower_clip_fraction'] if c in summary.columns]
if len(clip_cols) > 2:
    print('\nHard-clamp diagnostics:')
    display(summary[clip_cols].round(7))


In [ ]:
plt.figure(figsize=(9,6))
x = summary['cenn_state_mib_fp32']
y = summary['output_cosine']
plt.scatter(x, y)
for _,r in summary.iterrows():
    plt.annotate(f"{r['variant']} F={int(r['effective_feature_dim'])}", (r['cenn_state_mib_fp32'], r['output_cosine']), fontsize=8)
plt.xscale('log')
plt.xlabel('Constant CeNN state MiB/layer (FP32)')
plt.ylabel('Attention-output cosine')
plt.title('Quality/state-size trade-off')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## Optional full 30-layer confirmation
After identifying one or two promising variants, rerun only those settings across all layers. Do this **after** the representative-layer run; testing every poor variant across all 30 layers wastes GPU time. Edit `BEST_VARIANTS` and `BEST_FEATURE_DIMS` from the table above.


In [ ]:
RUN_ALL_LAYERS = False
BEST_VARIANTS = 'stable_orf,taylor2'
BEST_FEATURE_DIMS = '1024'

if RUN_ALL_LAYERS:
    full_dir = Path('/content/cenn-kernel-variants-all-layers')
    full_cmd = [
        sys.executable, str(REPO_DIR / 'scripts' / 'benchmark_cenn_kernel_variants.py'),
        '--base-model', 'HuggingFaceTB/SmolLM2-135M',
        '--context-length', str(CONTEXT_LENGTH),
        '--num-sequences', '2', '--layers', 'all',
        '--variants', BEST_VARIANTS, '--feature-dims', BEST_FEATURE_DIMS,
        '--output-dir', str(full_dir),
    ]
    subprocess.run(full_cmd, check=True)
    display(pd.read_csv(full_dir / 'kernel_variant_summary.csv').round(5))
else:
    print('Set RUN_ALL_LAYERS=True after choosing the promising variants.')
